---
jupyter: python3
format:
  html:
    code-fold: true
    code-tools: true
bibliography: ../../references.bib
author:
  - name: XXXX
    id: dw
    orcid: XXXX
    email: XXXX
    corresponding: true
    degrees: PhD
    affiliation: 
      - name: The Alan Turing Institute
        city: London
        country: United Kingdom
        url: www.turing.ac.uk

---

# Combine census, railspace, and StopsGB

This notebook shows how to combine the AddressGB, StopsGB, and Railspace datasets to produce the dataset used in the Beyond the Tracks chapter [add doi].

### Data inputs:
1. [AddressGB](https://doi.org/10.5281/zenodo.10473597) - Geo-coded historic census data (1851-1911)

2. [StopsGB](https://doi.org/10.23636/wvva-3d67) - Geo-located passenger railway station dataset

3. [Railspace (MapReader output)](https://doi.org/10.5281/zenodo.7147906) - Geo-located railway infrastructure (tracks and buildings)

### Data output:
1. BeyondtheTracks (DOI pending)

BeyondtheTracks is an enhanced version of AddressGB, which contains the following variables supplemented from StopsGB and Railspace for each street/address:

1. Identity and distance of nearest StopsGB station
2. Density of railspace within 100m (calculated as % of MapReader patches identified as containing railway tracks or buildings)

In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np
import re
from sklearn.neighbors import BallTree
import pathlib
import utils
import requests
import zipfile

## Download Data

Before running the code to combine the three datasets, make sure you have the relevant files downloaded. The code in this notebook assumes that the files are saved in the follow directory structure:

```bash
└── data
    ├── stopsgb
    │   └── StopsGB.tsv
    ├── mapreader
    │   ├── MapReader_Data_SIGSPATIAL_2022
    │   │   └── outputs
    │   │       ├── label_01_03
    │   │       │   └── pred_01_03_keep_01_0250.csv
    │   │       └── patches_all.csv

        │   ├── 1851_ew_geocode.txt
        │   ├── 1861_ew_geocode.txt
    
```

### AddressGB

Follow this [link](https://doi.org/10.5281/zenodo.10473597) to download the AddressGB data. To link AddressGB to StopsGB and Mapreader Railspace, we only need the geometry files from AddressGB, i.e. those with the suffix '_gb1900.tsv' or '_osopenroads.tsv'. The other AddressGB files are only needed when we want to link back to the original census data in [I-CeM](https://www.campop.geog.cam.ac.uk/research/projects/icem/). See other chapters in this book for how to do this. [add cross references to other chapters here]

### StopsGB

Follow this [link](https://doi.org/10.23636/wvva-3d67), to access the StopsGB dataset on the British Library Repository.

Download the `StopsGB.tsv` file.

### Mapreader Railspace

Follow this [link](https://doi.org/10.5281/zenodo.7147906) to download the railspace outputs from MapReader. Note, we only need 2 files: the file containing all patches (`patches_all.csv`) and the railway track and building patches (`pred_01_03_keep_01_0250.csv`). There are newer versions available [here]( https://doi.org/10.5281/zenodo.11241371) but these were produced after the Beyond the Tracks Chapter was written.

### Set Census Year and Country

The Beyond the Tracks dataset contains data for every census year 1851-1911 but the Beyond the Tracks Chapter only used 1901 census data. This was partly due to space constraints in the chapter itself. But it was also because 1901 was the optimal census year to analyse because it falls in the middle of the period c.1888-1914 that the 2nd Edition 6-inch Ordnance Survey maps were produced that were used to create the Railspace dataset.

Somethings to bear in mind if you use BeyondtheTracks for other years: 1891 and 1911 are probably perfectly ok to use, although 1891 is on the early side of the period when the maps underpinning the MapReader Railspace dataset were surveyed. Also, there is no Scottish data for 1911 in I-CeM (the census dataset that AddressGB links to).

1851 to 1881 predate the maps that the railspace dataset is based on. If you're working on these years, it's best to limit your analysis to distances to stations, since StopsGB covers this period.

In [3]:
censusvars = utils.set_census_vars("GB", 1901)

In [14]:
geoms = utils.set_geoms("gb1900")

In [4]:
def read_streetsgb(streetsgb_path, census_country, census_year, geom, ):
    if geom == "openroads":
        geom_file_modifier = "os"
    else:
        geom_file_modifier = ""
        
    if census_country == "GB":
        blockcols = [f"{geom}_uid", "Country"]
        gdf_list = []
        for country in ["EW", "scot"]:
            gdf = gdf_reader(streetsgb_path, geom, country, census_year, geom_file_modifier )
            gdf["Country"] = country
            # gdf = gdf.rename(columns = {f"{geom}_{country}_{census_year}" : f"{geom}_{census_country}_{census_year}"})
            gdf_list.append(gdf)

        streetsgb_gdf = pd.concat(gdf_list)
        
    elif census_country == "EW" or "scot":
        blockcols = [f"{geom}_uid", ]
        streetsgb_gdf = gdf_reader(streetsgb_path, geom, census_country, census_year, geom_file_modifier )

    else:
        raise ValueError()
    
    streetsgb_gdf = streetsgb_gdf[streetsgb_gdf["geometry"].is_empty == False].reset_index(drop=True).copy()
    return streetsgb_gdf, blockcols

def gdf_reader(streetsgb_path, geom, country, census_year, geom_file_modifier):
     streetsgb_file = pathlib.Path(streetsgb_path).joinpath(f"{country}_{census_year}_{geom_file_modifier}{geom}.tsv")
     if streetsgb_file.exists() == False:
         raise ValueError(streetsgb_file)
     else:
         
        # gdf = gpd.read_file(streetsgb_file)
        df = pd.read_csv(streetsgb_file, sep = "\t", usecols=[f"{geom}_uid", "geometry"], )
        # print(df)
        gdf = gpd.GeoDataFrame(df, geometry=gpd.GeoSeries.from_wkt(df['geometry']), crs="EPSG:27700")
        # gdf = gdf[[f"{geom}_{country}_{census_year}", "geometry"]].dropna(subset = ["geometry"]).copy() decide if need
     return gdf

## Read and process StopsGB (station data)

Read StopsGB, filter by confidence, cross_ref, ghost_entry, opening and closing year.

The first step is to read in the StopsGB data.

In [5]:
stopsgb_path = pathlib.Path("../../../data/stopsgb/")
stopsgb_path.mkdir(parents=True, exist_ok=True)

stopsgb_filename = "StopsGB.tsv"
stopsgb_file_path= stopsgb_path.joinpath(stopsgb_filename)

def read_stopsgb(file_path):
    stopsgb = pd.read_csv(file_path,
						sep='\t',
						usecols=['StationId',
						'Station',
						'Opening',
						'Closing',
						'cross_ref',
						'ghost_entry',
						'conf_station',
						'selected_entity_longitude',
						'selected_entity_latitude'])
    
    return stopsgb


if stopsgb_file_path.exists():
    stopsgb = read_stopsgb(stopsgb_file_path)
else:
    r = requests.get(f"https://bl.iro.bl.uk/downloads/e5594bc4-85f4-4205-8ccc-33e5e7105f07?locale=en", allow_redirects=True)

    with open(stopsgb_file_path, "wb") as f:
        f.write(r.content)

    stopsgb = read_stopsgb(stopsgb_file_path)

stopsgb

,StationId,Station,Opening,Closing,selected_entity_latitude,selected_entity_longitude,conf_station,cross_ref,ghost_entry
0,1,ABBEY TOWN,1856-09-03,1964-09-07,54.8459,-3.28742,0.59,False,False
1,2,ABBEY JUNCTION,1870-08-31,1921-05-20,54.8524,-3.27856,0.59,False,False
2,3,ABBEY JUNCTION,1870-08-08,1921-09-01,54.8524,-3.27856,0.59,False,False
3,4,ABBEY AND WEST DEREHAM,1882-08-01,1930-09-22,52.5704,0.4435,0.60,False,False
4,5,ABBEY FOREGATE,unknown,unknown,52.707285,-2.74329,0.59,True,False
...,...,...,...,...,...,...,...,...,...
12671,12672,YSTRAD CARDIGAN,unknown,unknown,unknown,unknown,0.00,True,False
12672,12673,YSTRAD MYNACH,1858-03-31,still open,51.6407,-3.2417,0.93,False,False
12673,12674,YSTRAD RHONDDA,1986-09-29,still open,51.6436,-3.4668,0.92,False,False
12674,12675,YSTRADGYNLAIS,1873-11-19,1932-09-12,51.7715,-3.7516,0.60,False,False


Next, we need to extract the opening and closing years of each station. Valid opening and closing dates are stored in the fromat 'yyyy-mm-dd'. Since we only want the year, we extract the first four characters or digits of the opening/closing fields. We also need to deal with unknown dates, which are recorded as 'unkn' in StopsGB. We find these and change them to `np.nan` values.

In [6]:
stopsgb['Opening_year'] = np.where(stopsgb['Opening'].str[0:4] == 'unkn',np.nan,stopsgb['Opening'].str[0:4])
stopsgb['Closing_year'] = np.where(stopsgb['Closing'].str[0:4] == 'unkn',np.nan,stopsgb['Closing'].str[0:4])

stopsgb['Opening_year'] = pd.to_numeric(stopsgb['Opening_year'], errors='coerce')
stopsgb['Closing_year'] = pd.to_numeric(stopsgb['Closing_year'], errors='coerce')

We then filter out stations on a series of criteria to ensure our dataset is as accurate as possible:
1. Exclude stations not open in specified census year by ensuring that the station opening date was before and closing date after our specified `census_year`
2. Exclude 'low confidence' stations. The `conf_station` field in StopsGB reports the confidence that the match is correct (between the Quicks word document and the corresponding Wikidata entry) - see the StopsGB documentation for further details. We only include stations where this confidence measure is greater than 50% (0.5)
3. Filter out other types of station entries, e.g. cross-references and ghost entries (see StopsGB documentaiton for further details)

In [7]:
stopsgb_filtered = stopsgb[(stopsgb['conf_station'] >= 0.5) 
							& (stopsgb['Opening_year'] <= censusvars.year)
							& ((stopsgb['Closing_year'] > censusvars.year) | (stopsgb['Closing'] == 'still open'))
							& (stopsgb['cross_ref'] == False)
							& (stopsgb['ghost_entry'] == False)].copy()

Convert our filtered StopsGB dataframe into a geopandas dataframe, so that we can peform geomtry operations on it (we'll need this when calculating the distance from streets in AddressGB to these stations). Change the Projection CRS to match AddressGB, which is in British National Grid (EPSG:27700). This makes sure that our distance calculations are in reported in meters.

In [8]:
stopsgb_gdf = gpd.GeoDataFrame(stopsgb_filtered,
								geometry=gpd.points_from_xy(stopsgb_filtered['selected_entity_longitude'], 
								stopsgb_filtered['selected_entity_latitude']),crs='EPSG:4326')

# Reproject for compatibility with AddressGB dataset
stopsgb_gdf = stopsgb_gdf.to_crs('EPSG:27700')

Here's a snippet of what the resulting dataset will look like. We've kept only the columns that are necessary for linking to AddressGB (this will reduce memory requirements).

In [9]:
stopsgb_gdf = stopsgb_gdf[['StationId','Station','geometry']]

stopsgb_gdf.head()

,StationId,Station,geometry
0,1,ABBEY TOWN,POINT (317428.186 550880.923)
1,2,ABBEY JUNCTION,POINT (318010.287 551593.76)
2,3,ABBEY JUNCTION,POINT (318010.287 551593.76)
3,4,ABBEY AND WEST DEREHAM,POINT (565696.082 299757.158)
9,10,ABBEY WOOD,POINT (547374.609 179028.254)


## Read and process MapReader Railspace Data

Read the rail insfrastructure patches from the Mapreader Railspace dataset. Downloads the data if not already downloaded.

In [10]:
mapreader_path = pathlib.Path("../../../data/mapreader/")
mapreader_path.mkdir(parents=True, exist_ok=True)

mapreader_ziparchive = "MapReader_Data_SIGSPATIAL_2022.zip"
mapreader_ziparchive_path = mapreader_path.joinpath(mapreader_ziparchive)

mapreader_rail_path = mapreader_path.joinpath("MapReader_Data_SIGSPATIAL_2022/outputs/label_01_03/pred_01_03_keep_01_0250.csv")
mapreader_all_path = mapreader_path.joinpath("MapReader_Data_SIGSPATIAL_2022/outputs/patches_all.csv")

def read_mapreader_data(mapreader_rail_filepath, mapreader_all_filepath):
    mapreader_rail = pd.read_csv(mapreader_rail_filepath,
                                    sep=',',
                                    usecols=['center_lon','center_lat','pred','conf'])
    
    mapreader_all = pd.read_csv(mapreader_all_filepath,
                                sep=',',
                                usecols=['center_lon','center_lat'])
    
    return mapreader_rail, mapreader_all


if mapreader_rail_path.exists():
    mapreader_rail, mapreader_all = read_mapreader_data(mapreader_rail_path, mapreader_all_path)

else:
    r = requests.get(f"https://zenodo.org/api/records/7147906/files/MapReader_Data_SIGSPATIAL_2022.zip/content", allow_redirects=True)
    with open(mapreader_ziparchive_path, "wb") as f:
        f.write(r.content)

    with zipfile.ZipFile(mapreader_ziparchive_path,"r") as zip_ref:
        zip_ref.extractall(mapreader_path)

    mapreader_rail, mapreader_all = read_mapreader_data(mapreader_rail_path, mapreader_all_path)

Convert the mapreader_rail dataframe to a geopandas dataframe to allow for geometry operations. Reproject it to match the CRS of AddressGB ('EPSG:27700')

In [11]:
mapreader_rail_gdf = gpd.GeoDataFrame(mapreader_rail,
                                        geometry=gpd.points_from_xy(mapreader_rail['center_lon'],
                                        mapreader_rail['center_lat']),
                                        crs='EPSG:4326')

mapreader_rail_gdf = mapreader_rail_gdf.to_crs('EPSG:27700')

Convert the mapreader_all dataframe to a geopandas dataframe to allow for geometry operations. Reproject it to match the CRS of AddressGB ('EPSG:27700')

In [12]:
mapreader_all_gdf = gpd.GeoDataFrame(mapreader_all,geometry=gpd.points_from_xy(mapreader_all['center_lon'], mapreader_all['center_lat']),crs='EPSG:4326')
mapreader_all_gdf = mapreader_all_gdf.to_crs('EPSG:27700')

In [13]:
# X_railspace = np.array(list(zip(mapreader_rail_gdf['coords1'],mapreader_rail_gdf['coords2'])))
X_railspace = np.array(list(zip(mapreader_rail_gdf.geometry.x,mapreader_rail_gdf.geometry.y)))
print(X_railspace.shape)
# X_all = np.array(list(zip(mapreader_all_gdf['coords1'],mapreader_all_gdf['coords2'])))
X_all = np.array(list(zip(mapreader_all_gdf.geometry.x,mapreader_all_gdf.geometry.y)))
print(X_all.shape)

(483278, 2)
(30490411, 2)


In [14]:
leafsize = 40
tree_railspace = BallTree(X_railspace, leaf_size=leafsize,metric='euclidean')
tree_all = BallTree(X_all, leaf_size=leafsize,metric='euclidean')

In [15]:
#this needs fixing because the street uids are not unique for GB.
def get_nearest_station(stopsgb, streetsgb, distance_col, geom, census_country, census_year, blockcols):

    dist2stopsgb_tmp = gpd.sjoin_nearest(stopsgb,streetsgb,how='right',distance_col='distance').drop(columns = ["index_left"]) # calcs nearest StopsGB station to each street
    dist2stopsgb_multi_only = dist2stopsgb_tmp[dist2stopsgb_tmp.duplicated(subset=blockcols, keep = False)]
    dist2stopsgb_multi_only = dist2stopsgb_multi_only.groupby(blockcols)["StationId"].apply(lambda x: x.to_list()).reset_index(name = "equidistant_stationids") # keep record of the stations that are equidistant to a street
    dist2stopsgb = pd.merge(left = dist2stopsgb_tmp, right = dist2stopsgb_multi_only, on = blockcols, how="left")

    dist2stopsgb = dist2stopsgb.drop_duplicates(subset= blockcols, keep = "first").copy()

    return dist2stopsgb.reset_index(drop=True)

def create_coords_field(df, geometry_field, new_coords_field, ):

    parse = lambda x: re.compile(r'\({1,2}(.*?)\)').findall(str(x))
    to_tuple = lambda x: tuple(map(float,x.split(' ')))
    to_coords = lambda r: [to_tuple(x) for i in r for x in i.split(', ')]

    df[new_coords_field] = df[geometry_field].apply(parse)
    df[new_coords_field] = df[new_coords_field].apply(to_coords)
    df[new_coords_field] = df[new_coords_field].apply(lambda r: [[x[0], x[1] ] for x in r])

    return df

In [16]:
def rail_density(x,tree_rail,tree_all,target_radius):
    #could add nan capture here rather than removing from whole dataframe
    # print(x)
    avg_rail =  tree_rail.query_radius(x, r=target_radius,count_only=True)
    avg_all = tree_all.query_radius(x, r=target_radius,count_only=True)

    # may need to sort out nans


    
    mean_val = np.mean(avg_rail / avg_all)
    stddev_val = np.std(avg_rail / avg_all)

    if mean_val == np.nan:
        print("meanval", x)
    elif stddev_val == np.nan:
        print("stdval", x)
    # return np.mean(avg_rail / avg_all),np.std(avg_rail / avg_all)
    return mean_val, stddev_val


def get_railspace_scores(streets_gdf, radius, railspace_patch_tree, all_patch_tree, ):
    streets_gdf[f'rail_density_{radius}m'], streets_gdf[f'rail_density_{radius}m_std'] = zip(*streets_gdf['coords'].apply(rail_density, 
                                                        tree_rail = railspace_patch_tree, 
                                                        tree_all = all_patch_tree,
                                                        target_radius = radius))
    return streets_gdf

In [18]:
addressgb_path = pathlib.Path("../../../data/addressgb/")
addressgb_path.mkdir(parents=True, exist_ok=True)

addressgb_ziparchive = "addressgb.zip"
addressgb_ziparchive_path = mapreader_path.joinpath(addressgb_ziparchive)

if addressgb_path.exists():
    pass
else:
    r = requests.get(f"https://zenodo.org/api/records/10473597/files/{addressgb_ziparchive}/content", allow_redirects=True)

    with open(addressgb_ziparchive_path, "wb") as f:
        f.write(r.content)

    with zipfile.ZipFile(addressgb_ziparchive_path,"r") as zip_ref:
        zip_ref.extractall(addressgb_path)


## Read and process AddressGB (census data) and calculate nearest station and railspace density

In [23]:
with np.errstate(invalid='ignore'):
    
    public_release = True

    target_geometries = ["gb1900", "openroads"] #calculate distances and densities for the 2 different geometries used in StreetsGB

    density_distances = [100, ] # set distances from street to calculate railspace densities (in meters)

    for geom in target_geometries:
        print(geom)
        (streetsgb_gdf, blockcols) = read_streetsgb(addressgb_path, censusvars.country, censusvars.year, geom, )


        print(streetsgb_gdf)
        dist2stopsgb = get_nearest_station(stopsgb_gdf, streetsgb_gdf, "distance", geom, censusvars.country, censusvars.year, blockcols )
        dist2stopsgb = create_coords_field(dist2stopsgb, "geometry", "coords", )

        # calc railspace densities at set distances
        for rad in density_distances:
            print(rad)
            dist2stopsgb = get_railspace_scores(dist2stopsgb, rad, tree_railspace, tree_all, )

        dist2stopsgb = dist2stopsgb.drop(columns = ["coords"])

        if public_release == True:
            dist2stopsgb = dist2stopsgb.drop(columns = ["geometry"])


        if censusvars.country == "GB":
            col_order = [f"{geom}_uid", "Country", "StationId", "equidistant_stationids", "distance", "rail_density_100m", "rail_density_100m_std"]
        else:
            col_order = [f"{geom}_uid", "StationId", "equidistant_stationids", "distance", "rail_density_100m", "rail_density_100m_std"]

        dist2stopsgb[col_order].to_csv(f'../../../data/{geom}_{censusvars.country}_{censusvars.year}_stations_and_raildensity.tsv',sep='\t',index=False)

gb1900
         gb1900_uid                       geometry Country
0                 0  POINT (524926.445 181641.475)      EW
1                 1  POINT (525500.238 181417.898)      EW
2                 2  POINT (525340.084 181780.915)      EW
3                 3  POINT (525510.673 182020.598)      EW
4                 4   POINT (525095.94 181170.202)      EW
...             ...                            ...     ...
1099686      231276  POINT (332817.733 734945.029)    scot
1099687      231277  POINT (334729.495 732683.297)    scot
1099688      231278  POINT (334729.055 734236.955)    scot
1099689      231279  POINT (334216.784 733500.161)    scot
1099690      231280  POINT (335604.757 735166.594)    scot

[1099691 rows x 3 columns]
100
